# 04 — Feature engineering

Builds the real per-cycle `FeatureObservation` rows via `build_feature_results` and inspects the 71-feature schema, the point-in-time selection rule, and how genuinely-missing values are carried rather than imputed.

**Data mode: REAL DATA.** Every figure below is computed from the genuine Bloomberg CSV exports for this strategy, through production AtlasQuant code. No synthetic, simulated, imputed, or approximated value appears anywhere in this notebook.

Those exports are gitignored, so they are absent in a fresh clone. When they are missing this notebook still executes top to bottom, but every computing cell prints an explicit *real data not present — cannot produce a genuine result* notice instead of a number. It never falls back to synthetic data: a plausible-looking figure derived from invented inputs is not a result (see `CLAUDE.md`).

In [ ]:
import _real_data as rd

print(rd.banner())
AVAILABLE = rd.data_available()


## The 71-feature schema

`FEATURE_NAMES` is the canonical column order — a model matrix's columns always come from this tuple, never from dict iteration order.

In [ ]:
from atlas_quant.strategies.multi_factor_ranking_ml.feature_domain import FEATURE_NAMES

blocks = {
    "1. fundamentals/market (fundamentals_quarterly.csv)": FEATURE_NAMES[:51],
    "2. carried over from the legacy feature export": FEATURE_NAMES[51:63],
    "3. locally computed (quarter_num, sector_enc)": FEATURE_NAMES[63:65],
    "4. market-wide macro, joined by available_date": FEATURE_NAMES[65:],
}
for label, names in blocks.items():
    print(f"{label}: {len(names)}")
    print("   ", ", ".join(names[:6]) + (", ..." if len(names) > 6 else ""))
print()
print("total:", len(FEATURE_NAMES))


## Building the real feature rows

One `FeatureObservation` per instrument per cycle, built from whichever fiscal history was most recently knowable as of that cycle's own cutoff. A row's own `quarter_end` frequently does *not* equal the cycle's calendar quarter — that is expected, not an error (most issuers use offset or 52/53-week fiscal years).

In [ ]:
if AVAILABLE:
    bundle = rd.slice_bundle()
    cycles = bundle.cycles

    print(f"{'cycle':<12} {'cutoff':<12} {'observations':>13} {'rejected':>9}")
    for cycle in cycles:
        result = bundle.feature_results[cycle.quarter_start]
        print(f"{str(cycle.quarter_start):<12} {str(cycle.cutoff):<12} "
              f"{len(result.observations):>13,} {len(result.rejected):>9,}")
else:
    bundle = None
    print(rd.DATA_UNAVAILABLE_MESSAGE)


## Why rows are rejected

Rejection happens only for genuine data insufficiency: no fundamentals row knowable yet as of the cutoff.

In [ ]:
if AVAILABLE:
    from collections import Counter

    cycle = bundle.cycles[-1]
    result = bundle.feature_results[cycle.quarter_start]
    reasons = Counter(rejection.reason.split("(")[0].strip() for rejection in result.rejected)
    print(f"cycle {cycle.quarter_start}: {len(result.rejected):,} rejected")
    for reason, count in reasons.most_common():
        print(f"  {count:>5,}  {reason}")
else:
    print(rd.DATA_UNAVAILABLE_MESSAGE)


## Point-in-time safety

Every observation for a cycle must have been knowable at that cycle's cutoff. This asserts it directly on the real rows rather than trusting it.

In [ ]:
if AVAILABLE:
    from datetime import datetime, time

    for cycle in bundle.cycles:
        cutoff_dt = datetime.combine(cycle.cutoff, time.max)
        observations = bundle.feature_results[cycle.quarter_start].observations
        assert all(obs.data_cutoff <= cutoff_dt for obs in observations)
        assert all(obs.filing_timestamp <= cutoff_dt for obs in observations)
        assert all(obs.quarter_end <= cycle.cutoff for obs in observations)
    print("OK: for every cycle, every observation's filing timestamp, quarter_end and")
    print("    data_cutoff are on or before that cycle's own cutoff. No lookahead.")
else:
    print(rd.DATA_UNAVAILABLE_MESSAGE)


## Fiscal quarter_end vs. strategy cohort

Confirms the two "quarter" concepts really are distinct in the real data — an exact match is used only to refine entry-timing precision, never as a requirement for a row to exist.

In [ ]:
if AVAILABLE:
    cycle = bundle.cycles[-1]
    observations = bundle.feature_results[cycle.quarter_start].observations
    lags = sorted((cycle.cutoff - obs.quarter_end).days for obs in observations)
    aligned = sum(1 for obs in observations if obs.quarter_end.month in (3, 6, 9, 12))
    print(f"cycle {cycle.quarter_start}, {len(observations):,} observations")
    print(f"  calendar-quarter-aligned fiscal quarter_end: {aligned:,} "
          f"({aligned / len(observations):.1%})")
    print(f"  age of the selected fiscal row at cutoff (days): "
          f"min {lags[0]}, median {lags[len(lags) // 2]}, max {lags[-1]}")
else:
    print(rd.DATA_UNAVAILABLE_MESSAGE)


## NaN handling

A legitimately unavailable feature is `float("nan")` in the observation and its name appears in `missing_features`. It is **never imputed** — gradient-boosted trees handle NaN natively, and imputing would invent a value the data does not contain.

In [ ]:
if AVAILABLE:
    import math

    cycle = bundle.cycles[-1]
    observations = bundle.feature_results[cycle.quarter_start].observations

    missing_counts = {name: 0 for name in FEATURE_NAMES}
    for obs in observations:
        for name in obs.missing_features:
            missing_counts[name] += 1

    ranked = sorted(missing_counts.items(), key=lambda item: -item[1])
    print(f"cycle {cycle.quarter_start}, {len(observations):,} observations")
    print()
    print("most-missing features:")
    for name, count in ranked[:12]:
        print(f"  {name:<40} {count:>6,}  ({count / len(observations):>6.1%})")
    print()
    complete = sum(1 for name, count in ranked if count == 0)
    print(f"features with no missing values at all: {complete} of {len(FEATURE_NAMES)}")
    print()
    per_row = sorted(len(obs.missing_features) for obs in observations)
    print(f"missing features per row: min {per_row[0]}, "
          f"median {per_row[len(per_row) // 2]}, max {per_row[-1]}")
    print()
    example = observations[0]
    values = example.to_model_row()
    print(f"example row: {example.instrument_id.symbol}  sector={example.sector}")
    print("  first 6 feature values:",
          [("nan" if math.isnan(v) else round(v, 4)) for v in values[:6]])
else:
    print(rd.DATA_UNAVAILABLE_MESSAGE)


## Findings

- The modern cross-section yields roughly 1,400 usable feature rows per quarterly cycle out of a 1,520-ticker universe; the rest are rejected for having no fundamentals row knowable yet at the cutoff.
- The point-in-time assertion holds on the real data for every cycle in the slice: no observation uses a fiscal row or a cutoff later than its own cycle's.
- Missingness is concentrated in a minority of features; most rows carry a handful of NaNs, and those NaNs reach the model as NaNs.

## Limitations

- Two of the 71 features are not genuinely point-in-time: `sector_enc` derives from the present-day GICS snapshot (notebook 03), and features carried over from the legacy export inherit whatever timing that export used.
- Feature *values* are taken as delivered by the provider; this notebook verifies their timing and completeness, not their correctness.
- The slice is 14 recent cycles. Feature availability in the 1980s–1990s is materially worse and is not exercised here.